# Q-Learning Debugging Practice — Delivery Robot

### Instructions
1. Run the program and identify the errors.
2. Correct the errors without rewriting the program from scratch.
3. Execute the corrected program.
4. Verify that the robot learns a successful route to the delivery point.


In [1]:
import numpy as np
import random
import gymnasium 

# ============================================================
# DELIVERY ROBOT USING Q-LEARNING
# FrozenLake 4x4 is interpreted as a warehouse environment.
#
# S = Robot starting point
# F = Safe warehouse floor
# H = Blocked/unsafe location
# G = Delivery point
# ============================================================

env = gymnasium.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True
)

state_space = env.observation_space.n
action_space = env.action_space.n
Q_table = np.zeros((action_space, state_space))

learning_rate = 7.0
discount_factor = 0.95

max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.001

n_training_episodes = 3000
max_steps = 100

# Epsilon-greedy action selection
def choose_action(Q_table, state, epsilon):
    if random.uniform(0, 1) > epsilon:
        action = env.action_space.sample()
    else:
        action = np.argmax(Q_table[:, state])
    return action

# Training
for episode in range(n_training_episodes + 1):

    state, info = env.reset()
    total_reward = 0

    epsilon = max_epsilon + (
        max_epsilon - min_epsilon
    ) * np.exp(-decay_rate * episode)

    for step in range(max_steps):

        action = choose_action(Q_table, state, epsilon)

        next_state, reward, terminated, truncated, info = env.step(action)

        best_next_q = np.max(Q_table[next_state])

        if terminated and truncated:
            target = reward
        else:
            target = reward + learning_rate * best_next_q

        Q_table[next_state, action] = (
            Q_table[state, action]
            + learning_rate * (
                target - Q_table[state, action]
            )
        )

        total_reward += reward
        state == next_state

        if terminated or truncated:
            break

print("\nTraining completed.")
print("Learned Q-table:")
print(np.round(Q_table, 3))

action_symbols = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

grid = env.unwrapped.desc

print("\nLearned Delivery Robot Policy:")
print("--------------------------------")

for row in range(4):
    line = ""

    for col in range(4):
        state = row * 4 + col
        cell = grid[row][col].decode()

        if cell == "S":
            symbol = "S"
        elif cell == "G":
            symbol = "G"
        elif cell == "H":
            symbol = "X"
        else:
            best_action = np.argmax(Q_table[state])

            if np.min(Q_table[state]) == 0:
                symbol = "."
            else:
                symbol = action_symbols[best_action]

        line += f"[ {symbol} ] "

    print(line)

# Test the trained robot
print("\nTesting the trained delivery robot...")

state, info = env.reset()
path = [state]
total_reward = 0

for step in range(max_steps):

    action = np.argmax(Q_table[state])

    next_state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    state = next_state
    path.append(state)

    if terminated or truncated:
        break

print("Path followed:", path)
print("Total reward:", total_reward)

if truncated and reward == 1:
    print("Delivery successful!")
else:
    print("Delivery unsuccessful.")

env.close()


IndexError: index 4 is out of bounds for axis 0 with size 4

In [2]:
import numpy as np
import random
import gymnasium

# ============================================================
# DELIVERY ROBOT USING Q-LEARNING
# FrozenLake 4x4 is interpreted as a warehouse environment.
#
# S = Robot starting point
# F = Safe warehouse floor
# H = Blocked/unsafe location
# G = Delivery point
# ============================================================

env = gymnasium.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True
)

state_space = env.observation_space.n
action_space = env.action_space.n

# Q-table: rows = states, columns = actions
Q_table = np.zeros((state_space, action_space))

learning_rate = 0.1
discount_factor = 0.95

max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.001

n_training_episodes = 3000
max_steps = 100


# ============================================================
# Epsilon-greedy action selection
# ============================================================

def choose_action(Q_table, state, epsilon):

    # Explore
    if random.uniform(0, 1) < epsilon:
        action = env.action_space.sample()

    # Exploit
    else:
        action = np.argmax(Q_table[state])

    return action


# ============================================================
# Training
# ============================================================

for episode in range(n_training_episodes):

    state, info = env.reset()
    total_reward = 0

    # Epsilon decreases from 1.0 to 0.05
    epsilon = min_epsilon + (
        max_epsilon - min_epsilon
    ) * np.exp(-decay_rate * episode)

    for step in range(max_steps):

        action = choose_action(Q_table, state, epsilon)

        next_state, reward, terminated, truncated, info = env.step(action)

        # Best future Q-value
        best_next_q = np.max(Q_table[next_state])

        # Q-learning target
        if terminated or truncated:
            target = reward
        else:
            target = reward + discount_factor * best_next_q

        # Update Q-value
        Q_table[state, action] = (
            Q_table[state, action]
            + learning_rate * (
                target - Q_table[state, action]
            )
        )

        total_reward += reward

        # Move to next state
        state = next_state

        if terminated or truncated:
            break


print("\nTraining completed.")
print("Learned Q-table:")
print(np.round(Q_table, 3))


# ============================================================
# Display learned policy
# ============================================================

action_symbols = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

grid = env.unwrapped.desc

print("\nLearned Delivery Robot Policy:")
print("--------------------------------")

for row in range(4):

    line = ""

    for col in range(4):

        state = row * 4 + col
        cell = grid[row][col].decode()

        if cell == "S":
            symbol = "S"

        elif cell == "G":
            symbol = "G"

        elif cell == "H":
            symbol = "X"

        else:
            best_action = np.argmax(Q_table[state])
            symbol = action_symbols[best_action]

        line += f"[ {symbol} ] "

    print(line)


# ============================================================
# Test trained robot
# ============================================================

print("\nTesting the trained delivery robot...")

state, info = env.reset()

path = [state]
total_reward = 0

for step in range(max_steps):

    # Always exploit learned policy
    action = np.argmax(Q_table[state])

    next_state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

    state = next_state
    path.append(state)

    if terminated or truncated:
        break


print("Path followed:", path)
print("Total reward:", total_reward)

if terminated and reward == 1:
    print("Delivery successful!")
else:
    print("Delivery unsuccessful.")

env.close()


Training completed.
Learned Q-table:
[[0.193 0.184 0.19  0.177]
 [0.091 0.071 0.1   0.162]
 [0.137 0.133 0.125 0.129]
 [0.076 0.045 0.053 0.119]
 [0.223 0.139 0.171 0.124]
 [0.    0.    0.    0.   ]
 [0.15  0.085 0.135 0.052]
 [0.    0.    0.    0.   ]
 [0.181 0.236 0.261 0.3  ]
 [0.277 0.434 0.232 0.263]
 [0.35  0.401 0.368 0.137]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.232 0.415 0.605 0.384]
 [0.536 0.69  0.812 0.561]
 [0.    0.    0.    0.   ]]

Learned Delivery Robot Policy:
--------------------------------
[ S ] [ ↑ ] [ ← ] [ ↑ ] 
[ ← ] [ X ] [ ← ] [ X ] 
[ ↑ ] [ ↓ ] [ ↓ ] [ X ] 
[ X ] [ → ] [ → ] [ G ] 

Testing the trained delivery robot...
Path followed: [0, 0, 0, 0, 0, 4, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 0, 0, 0, 0, 0, 0, 4, 8, 8, 4, 0, 0, 0, 0, 4, 0, 0, 0, 4, 0, 4, 0, 0, 4, 8, 9, 10, 14, 10, 11]
Total reward: 0
Delivery unsuccessful.


In [3]:
import numpy as np
import random
import gymnasium 

# ============================================================
# DELIVERY ROBOT USING Q-LEARNING
# FrozenLake 4x4 is interpreted as a warehouse environment.
#
# S = Robot starting point
# F = Safe warehouse floor
# H = Blocked/unsafe location
# G = Delivery point
# ============================================================

env = gymnasium.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True
)

state_space = env.observation_space.n
action_space = env.action_space.n
Q_table = np.zeros((action_space, state_space))

learning_rate = 0.7
discount_factor = 0.95

max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.001

n_training_episodes = 3000
max_steps = 100

# Epsilon-greedy action selection
def choose_action(Q_table, state, epsilon):
    if random.uniform(0, 1) < epsilon:
        action = env.action_space.sample()
    else:
        action = np.argmax(Q_table[:, state])
    return action

# Training
for episode in range(n_training_episodes + 1):

    state, info = env.reset()
    total_reward = 0

    epsilon = min_epsilon + (
        max_epsilon - min_epsilon
    ) * np.exp(-decay_rate * episode)

    for step in range(max_steps):

        action = choose_action(Q_table, state, epsilon)

        next_state, reward, terminated, truncated, info = env.step(action)

        best_next_q = np.max(Q_table[:, next_state])

        if terminated or truncated:
            target = reward
        else:
            target = reward + discount_factor * best_next_q

        Q_table[action, state] = (
            Q_table[action, state]
            + learning_rate * (
                target - Q_table[action, state]
            )
        )

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

print("\nTraining completed.")
print("Learned Q-table:")
print(np.round(Q_table, 3))

action_symbols = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

grid = env.unwrapped.desc

print("\nLearned Delivery Robot Policy:")
print("--------------------------------")

for row in range(4):
    line = ""

    for col in range(4):
        state = row * 4 + col
        cell = grid[row][col].decode()

        if cell == "S":
            symbol = "S"
        elif cell == "G":
            symbol = "G"
        elif cell == "H":
            symbol = "X"
        else:
            best_action = np.argmax(Q_table[:, state])

            if np.min(Q_table[:, state]) == 0:
                symbol = "."
            else:
                symbol = action_symbols[best_action]

        line += f"[ {symbol} ] "

    print(line)

# Test the trained robot
print("\nTesting the trained delivery robot...")

state, info = env.reset()
path = [state]
total_reward = 0

for step in range(max_steps):

    action = np.argmax(Q_table[:, state])

    next_state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    state = next_state
    path.append(state)

    if terminated or truncated:
        break

print("Path followed:", path)
print("Total reward:", total_reward)

if terminated and reward == 1:
    print("Delivery successful!")
else:
    print("Delivery unsuccessful.")

env.close()



Training completed.
Learned Q-table:
[[0.275 0.064 0.023 0.005 0.37  0.    0.479 0.    0.08  0.356 0.649 0.
  0.    0.04  0.674 0.   ]
 [0.137 0.015 0.034 0.024 0.006 0.    0.005 0.    0.098 0.54  0.016 0.
  0.    0.497 0.91  0.   ]
 [0.148 0.02  0.025 0.009 0.083 0.    0.007 0.    0.268 0.473 0.101 0.
  0.    0.606 0.454 0.   ]
 [0.224 0.089 0.035 0.03  0.104 0.    0.    0.    0.516 0.134 0.041 0.
  0.    0.608 0.614 0.   ]]

Learned Delivery Robot Policy:
--------------------------------
[ S ] [ ↑ ] [ ↑ ] [ ↑ ] 
[ ← ] [ X ] [ ← ] [ X ] 
[ ↑ ] [ ↓ ] [ ← ] [ X ] 
[ X ] [ ↑ ] [ ↓ ] [ G ] 

Testing the trained delivery robot...
Path followed: [0, 0, 4, 0, 0, 4, 4, 0, 0, 4, 8, 4, 0, 0, 4, 8, 4, 8, 9, 13, 12]
Total reward: 0
Delivery unsuccessful.


In [8]:
import numpy as np
import random
import gymnasium 

# ============================================================
# DELIVERY ROBOT USING Q-LEARNING
# FrozenLake 4x4 is interpreted as a warehouse environment.
#
# S = Robot starting point
# F = Safe warehouse floor
# H = Blocked/unsafe location
# G = Delivery point
# ============================================================

env = gymnasium.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True
)

state_space = env.observation_space.n
action_space = env.action_space.n
Q_table = np.zeros((action_space, state_space))

learning_rate = 0.1
discount_factor = 0.95

max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.001

n_training_episodes = 3000
max_steps = 100

# Epsilon-greedy action selection
def choose_action(Q_table, state, epsilon):
    if random.uniform(0, 1) < epsilon:
        action = env.action_space.sample()
    else:
        action = np.argmax(Q_table[:, state])
    return action

# Training
for episode in range(n_training_episodes + 1):

    state, info = env.reset()
    total_reward = 0

    epsilon = min_epsilon + (
        max_epsilon - min_epsilon
    ) * np.exp(-decay_rate * episode)

    for step in range(max_steps):

        action = choose_action(Q_table, state, epsilon)

        next_state, reward, terminated, truncated, info = env.step(action)

        best_next_q = np.max(Q_table[:, next_state])

        if terminated or truncated:
            target = reward
        else:
            target = reward + discount_factor * best_next_q

        Q_table[action, state] = (
            Q_table[action, state]
            + learning_rate * (
                target - Q_table[action, state]
            )
        )

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

print("\nTraining completed.")
print("Learned Q-table:")
print(np.round(Q_table, 3))

action_symbols = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

grid = env.unwrapped.desc

print("\nLearned Delivery Robot Policy:")
print("--------------------------------")

for row in range(4):
    line = ""

    for col in range(4):
        state = row * 4 + col
        cell = grid[row][col].decode()

        if cell == "S":
            symbol = "S"
        elif cell == "G":
            symbol = "G"
        elif cell == "H":
            symbol = "X"
        else:
            best_action = np.argmax(Q_table[:, state])

            if np.min(Q_table[:, state]) == 0:
                symbol = "."
            else:
                symbol = action_symbols[best_action]

        line += f"[ {symbol} ] "

    print(line)

# Test the trained robot
print("\nTesting the trained delivery robot...")

state, info = env.reset()
path = [state]
total_reward = 0

for step in range(max_steps):

    action = np.argmax(Q_table[:, state])

    next_state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    state = next_state
    path.append(state)

    if terminated or truncated:
        break

print("Path followed:", path)
print("Total reward:", total_reward)

if terminated and reward == 1:
    print("Delivery successful!")
else:
    print("Delivery unsuccessful.")

env.close()



Training completed.
Learned Q-table:
[[0.233 0.083 0.177 0.078 0.288 0.    0.137 0.    0.183 0.254 0.437 0.
  0.    0.288 0.533 0.   ]
 [0.196 0.084 0.125 0.062 0.149 0.    0.1   0.    0.191 0.487 0.3   0.
  0.    0.342 0.798 0.   ]
 [0.192 0.095 0.122 0.076 0.14  0.    0.256 0.    0.276 0.316 0.217 0.
  0.    0.623 0.596 0.   ]
 [0.176 0.171 0.107 0.121 0.159 0.    0.038 0.    0.372 0.247 0.151 0.
  0.    0.318 0.71  0.   ]]

Learned Delivery Robot Policy:
--------------------------------
[ S ] [ ↑ ] [ ← ] [ ↑ ] 
[ ← ] [ X ] [ → ] [ X ] 
[ ↑ ] [ ↓ ] [ ← ] [ X ] 
[ X ] [ → ] [ ↓ ] [ G ] 

Testing the trained delivery robot...
Path followed: [0, 4, 0, 4, 8, 4, 4, 8, 9, 8, 4, 4, 0, 4, 0, 0, 0, 4, 8, 9, 13, 14, 13, 14, 15]
Total reward: 1
Delivery successful!
